# Part 3 — Multi-Agent Supervisor

Revenue Agent + Expenditure Agent (`agents.py`) under a LangGraph supervisor (`part3_supervisor.py`,
`langgraph_supervisor.create_supervisor`). See `README.md` for the full writeup: architecture,
assumptions, and a real routing-quality finding from development (a demo query initially only
invoked one agent due to accidental context overlap between the two agents' page scoping).

## Sub-agents, verified individually before wiring the supervisor

Isolating failure points: confirm each agent answers correctly on its own before testing the
supervisor's routing on top of them.

In [1]:
from agents import build_revenue_agent, build_expenditure_agent

revenue_agent = build_revenue_agent()
r = revenue_agent.invoke({"messages": [{"role": "user", "content": "What is the largest single source of government revenue in FY2024, and what is its amount?"}]})
print("REVENUE AGENT (standalone):")
print(r["messages"][-1].content)

REVENUE AGENT (standalone):
Based on Table 2.1 in the FY2024 Budget, the largest single source of government revenue in FY2024 is **Corporate Income Tax at $28.03 billion**.

This is followed by Personal Income Tax at $18.07 billion and Goods and Services Tax at $19.39 billion. However, Corporate Income Tax remains the single largest revenue source for FY2024.


In [2]:
expenditure_agent = build_expenditure_agent()
r = expenditure_agent.invoke({"messages": [{"role": "user", "content": "How much is the Future Energy Fund being topped up by, and what will the money be used for?"}]})
print("EXPENDITURE AGENT (standalone):")
print(r["messages"][-1].content)

EXPENDITURE AGENT (standalone):
Based on the expenditure context, the **Future Energy Fund is being topped up by $5.0 billion**.

The money will be used to **invest in critical infrastructure for the energy transition**.

This is outlined in the Special Transfers section of Budget 2024, where it states: "The Government will establish the Future Energy Fund with an initial injection of $5.0 billion to invest in critical infrastructure for the energy transition."


## Supervisor: assignment's exact required query

"What are the key government revenue streams, and how will the Budget for the Future Energy Fund
be supported?" — the trace below shows the supervisor's actual routing decisions, not an assumed
or asserted mechanism: question -> routes to revenue_agent -> tool call -> revenue synthesis ->
back to supervisor -> routes to expenditure_agent -> tool call -> expenditure synthesis -> back to
supervisor -> final comprehensive synthesis.

In [3]:
from part3_supervisor import build_supervisor, run_query, print_trace, DEMO_QUERIES

app = build_supervisor()
q1_label = "Q1 (assignment's exact query, dual-agent)"
answer, trace = run_query(app, DEMO_QUERIES[q1_label])
print("TRACE:")
print_trace(trace)
print("\nAGENTS INVOKED:", sorted(set(t["actor"] for t in trace) - {"user", "supervisor"}))
print("\nFINAL ANSWER:\n", answer)

TRACE:
[1] HumanMessage   actor=user                     What are the key government revenue streams, and how will the Budget for the Future Energy Fund be supported?
[2] AIMessage      actor=supervisor               thinking: {'signature': 'EsUCCpABCBEYAipAvHeC62NG/hVCq6VPJ8kyamwZUiKKtVg4AsMl+9B0WhF3z9ByI9ZsUjcUnyYYJrfexQY0qG842FD51+iFORHz8DIPY | tool_use: {'id': 'toolu_01F6foxmSz1D7AMqh2gbNwAk', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'transfer_to_revenue_agent',
[3] ToolMessage    actor=transfer_to_revenue_agent Successfully transferred to revenue_agent
[4] AIMessage      actor=revenue_agent            text: I'll help you with information about government revenue streams. Let me retrieve the relevant context. | tool_use: {'id': 'toolu_01E6VkdR8w7KVCgMqRynMYhn', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'revenue_context', 'type': '
[5] ToolMessage    actor=revenue_context          --- PAGE 5 ---
MINISTRY OF FINANCE 
 
5 
 
01 Update on Financial Year 2023 
 
1.1 


## Verification against ground truth

- Future Energy Fund must be stated as **$5.0 billion**, with the "critical infrastructure for the
  energy transition" purpose from page 18 — not just a bare number.
- Both `revenue_agent` and `expenditure_agent` must appear in the trace's actor set (verified from
  the trace structure itself, not inferred from the answer sounding plausible).
- Revenue streams named in the answer must match the real 12-item Operating Revenue tax list from
  Part 1 (no invented categories).

In [4]:
actors = set(t["actor"] for t in trace)
checks = {
    "Future Energy Fund states $5.0 billion": "5.0 billion" in answer or "$5.0" in answer,
    "Future Energy Fund purpose (energy transition) present": "energy transition" in answer.lower(),
    "revenue_agent invoked (from trace)": "revenue_agent" in actors,
    "expenditure_agent invoked (from trace)": "expenditure_agent" in actors,
}
for label, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {label}")

PASS  Future Energy Fund states $5.0 billion
PASS  Future Energy Fund purpose (energy transition) present
PASS  revenue_agent invoked (from trace)
PASS  expenditure_agent invoked (from trace)


## Various queries demonstrating collaborative routing

Four queries designed to prove the supervisor genuinely **routes** (not reflexively calling both
agents every time) as well as genuinely **collaborates** when a query needs both:
1. The assignment's exact query (dual-agent) — run above.
2. Revenue-only query — should invoke only `revenue_agent`.
3. Expenditure-only query — should invoke only `expenditure_agent`.
4. A second, differently-phrased dual-agent query — should invoke both again, confirming query 1
   wasn't a one-off fluke.

In [5]:
import json

all_results = {q1_label: {"query": DEMO_QUERIES[q1_label], "trace": trace, "final_answer": answer}}

for label, query in DEMO_QUERIES.items():
    if label == q1_label:
        continue
    print(f"\n{'=' * 80}\n{label}\nQuery: {query}\n{'=' * 80}")
    a, t = run_query(app, query)
    print("\nAGENTS INVOKED:", sorted(set(x["actor"] for x in t) - {"user", "supervisor"}))
    print("\nFINAL ANSWER:\n", a)
    all_results[label] = {"query": query, "trace": t, "final_answer": a}

with open("trace.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)
print("\nWrote trace.json with all 4 demo queries.")


Q2 (revenue-only, tests selective routing)
Query: What is the largest single source of government revenue?



AGENTS INVOKED: ['revenue_agent', 'revenue_context', 'transfer_back_to_supervisor', 'transfer_to_revenue_agent']

FINAL ANSWER:
 ## Summary

**The largest single source of government revenue is Corporate Income Tax.**

Key figures from the Ministry of Finance data:

- **FY2023 (Revised):** Corporate Income Tax collections were **$28.4 billion**, representing **27.2%** of Operating Revenue — by far the largest component.
- **FY2024 (Estimated):** Corporate Income Tax is projected at **$28.03 billion**, still the top revenue source, though slightly lower than FY2023 (a decrease of $0.35 billion, or -1.2%).

For context, the next largest sources of revenue are:
1. **Personal Income Tax** – $17.53 billion (16.8% of revenue) in FY2023
2. **Goods and Services Tax** – $16.36 billion (15.7% of revenue) in FY2023

Corporate Income Tax revenue rose sharply in FY2023 due to stronger-than-expected economic growth in 2022, which continues to make it the dominant contributor to total government ope


AGENTS INVOKED: ['expenditure_agent', 'expenditure_context', 'transfer_back_to_supervisor', 'transfer_to_expenditure_agent']

FINAL ANSWER:
 ## Summary: GST Voucher Fund Top-up

**Amount:** $6.0 billion (this is the largest single fund top-up among all the Endowment and Trust Fund top-ups in Budget 2024, which total $20.4 billion overall)

**Reason:** The top-up is meant **to meet the steady state cashflow needs for the enhanced permanent GST Voucher scheme**. In other words, the Government is injecting this sum into the fund to ensure it can sustainably finance the GST Voucher payouts to Singaporeans on an ongoing basis, now that the scheme has been made permanent and enhanced.

For context, this $6.0 billion top-up is part of a broader set of Endowment and Trust Fund injections in Budget 2024, which also include top-ups to the Future Energy Fund ($5.0 billion), Edusave Endowment Fund ($2.0 billion), Financial Sector Development Fund ($2.0 billion), National Productivity Fund ($2.0 b


AGENTS INVOKED: ['expenditure_agent', 'expenditure_context', 'revenue_agent', 'revenue_context', 'transfer_back_to_supervisor', 'transfer_to_expenditure_agent', 'transfer_to_revenue_agent']

FINAL ANSWER:
 ## Comprehensive Answer

**1. Largest Source of Tax Revenue:**
**Corporate Income Tax** is the largest source of tax revenue for the Singapore government. According to the budget document:
- Estimated FY2024: **$28.03 billion**
- It represented **27.2%** of total Operating Revenue in FY2023 — the single largest component, ahead of Personal Income Tax (16.8%) and Goods and Services Tax (15.7%).

**2. Future Energy Fund Infrastructure Spending Purpose:**
According to the budget document, the Government will establish the **Future Energy Fund with an initial injection of $5.0 billion** specifically **to invest in critical infrastructure for the energy transition**. This fund represents one of the largest new Top-ups to Endowment and Trust Funds in Budget 2024, reflecting the government

## Routing-pattern summary

Expected pattern: dual / revenue-only / expenditure-only / dual. Confirms the supervisor's
routing decisions are genuine and query-dependent, not hardcoded or reflexive.

In [6]:
for label, result in all_results.items():
    agents_used = sorted(set(t["actor"] for t in result["trace"]) & {"revenue_agent", "expenditure_agent"})
    print(f"{label}: {agents_used}")

Q1 (assignment's exact query, dual-agent): ['expenditure_agent', 'revenue_agent']
Q2 (revenue-only, tests selective routing): ['revenue_agent']
Q3 (expenditure-only, tests selective routing): ['expenditure_agent']
Q4 (second dual-agent query, different phrasing): ['expenditure_agent', 'revenue_agent']
